<a href="https://colab.research.google.com/github/hs380102-blip/Python-projects/blob/main/campus_food_delivery_colab_(7).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Campus Food Delivery Agent — DIT-inspired 3D Colab Dashboard

Open this `.ipynb` using Colab’s **File → Upload notebook**. Do not paste the notebook’s raw JSON into a Python cell.

Run all cells from top to bottom. Add an order, preview the path selected for its priority, compare NORMAL/HIGH/URGENT route choices and alternative ETAs, then play the small robot animation. The 3D viewer is larger and includes a high-resolution PNG export button in its camera menu.

Map geometry, robot speed, crowd delays, and dispatch waits are illustrative demo values. Replace them with measured campus data before relying on the estimates.

In [ ]:
%pip -q install matplotlib ipywidgets plotly


In [ ]:
"""SQLite storage for the campus food-delivery project.

Run this file directly to create the database, add a sample order, and display
the saved order history:
    python food_order_database.py

The database file (food_delivery.db) is created beside this Python file.
Menu prices and menu entries are sample data; change them to match your cafe.
"""

from contextlib import contextmanager
from datetime import datetime
import sqlite3
from pathlib import Path


DEFAULT_MENU = [
    ("Veg Sandwich", "Snacks", 80.00),
    ("Paneer Roll", "Snacks", 110.00),
    ("Veg Momos", "Snacks", 90.00),
    ("Masala Maggi", "Meals", 70.00),
    ("Cold Coffee", "Drinks", 60.00),
]

ORDER_STATUSES = {
    "PENDING",
    "PREPARING",
    "OUT_FOR_DELIVERY",
    "DELIVERED",
    "CANCELLED",
}


class FoodOrderDatabase:
    """Create and manage menu items, customer orders, and ordered items."""

    def __init__(self, database_path=None):
        if database_path is None:
            module_file = globals().get("__file__")
            if module_file:
                database_path = Path(module_file).with_name("food_delivery.db")
            else:
                database_path = Path.cwd() / "food_delivery.db"
        self.database_path = Path(database_path)

    @contextmanager
    def _connect(self):
        connection = sqlite3.connect(self.database_path)
        connection.row_factory = sqlite3.Row
        connection.execute("PRAGMA foreign_keys = ON")
        try:
            yield connection
            connection.commit()
        except Exception:
            connection.rollback()
            raise
        finally:
            connection.close()

    def initialize(self):
        """Create tables and insert the example menu if it is not present."""
        with self._connect() as connection:
            connection.executescript(
                """
                CREATE TABLE IF NOT EXISTS food_items (
                    food_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    name TEXT NOT NULL UNIQUE,
                    category TEXT NOT NULL,
                    price REAL NOT NULL CHECK (price >= 0),
                    is_available INTEGER NOT NULL DEFAULT 1
                        CHECK (is_available IN (0, 1))
                );

                CREATE TABLE IF NOT EXISTS orders (
                    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    customer_name TEXT NOT NULL,
                    cafeteria TEXT NOT NULL,
                    destination TEXT NOT NULL,
                    priority TEXT NOT NULL DEFAULT 'NORMAL',
                    status TEXT NOT NULL DEFAULT 'PENDING',
                    created_at TEXT NOT NULL,
                    total REAL NOT NULL DEFAULT 0 CHECK (total >= 0)
                );

                CREATE TABLE IF NOT EXISTS order_items (
                    order_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    order_id INTEGER NOT NULL,
                    food_id INTEGER NOT NULL,
                    quantity INTEGER NOT NULL CHECK (quantity > 0),
                    unit_price REAL NOT NULL CHECK (unit_price >= 0),
                    FOREIGN KEY (order_id) REFERENCES orders(order_id)
                        ON DELETE CASCADE,
                    FOREIGN KEY (food_id) REFERENCES food_items(food_id)
                );
                """
            )
            connection.executemany(
                """
                INSERT OR IGNORE INTO food_items (name, category, price)
                VALUES (?, ?, ?)
                """,
                DEFAULT_MENU,
            )

    def list_menu(self, available_only=True):
        """Return menu rows as dictionaries, ordered by food ID."""
        query = "SELECT food_id, name, category, price, is_available FROM food_items"
        if available_only:
            query += " WHERE is_available = 1"
        query += " ORDER BY food_id"

        with self._connect() as connection:
            rows = connection.execute(query).fetchall()
            return [dict(row) for row in rows]

    def add_menu_item(self, name, category, price):
        """Add a menu item and return its ID."""
        name = name.strip()
        category = category.strip()
        if not name or not category:
            raise ValueError("Food name and category cannot be empty.")
        if price < 0:
            raise ValueError("Price cannot be negative.")

        with self._connect() as connection:
            cursor = connection.execute(
                """
                INSERT INTO food_items (name, category, price)
                VALUES (?, ?, ?)
                """,
                (name, category, float(price)),
            )
            return cursor.lastrowid

    def set_item_availability(self, food_id, available):
        """Mark a menu item available or unavailable."""
        with self._connect() as connection:
            cursor = connection.execute(
                "UPDATE food_items SET is_available = ? WHERE food_id = ?",
                (1 if available else 0, food_id),
            )
            if cursor.rowcount == 0:
                raise ValueError(f"No food item has ID {food_id}.")

    def create_order(
        self,
        customer_name,
        cafeteria,
        destination,
        selections,
        priority="NORMAL",
    ):
        """Save an order and its item rows as one transaction.

        `selections` is an iterable of (food_id, quantity) pairs. The price is
        copied into order_items so later menu price changes do not alter old
        orders.
        """
        customer_name = customer_name.strip()
        cafeteria = cafeteria.strip().upper()
        destination = destination.strip().upper()
        priority = priority.strip().upper()

        if not customer_name:
            raise ValueError("Customer name cannot be empty.")
        if not cafeteria or not destination:
            raise ValueError("Cafeteria and destination are required.")
        if priority not in {"NORMAL", "HIGH", "URGENT"}:
            raise ValueError("Priority must be NORMAL, HIGH, or URGENT.")

        quantities = {}
        for food_id, quantity in selections:
            food_id = int(food_id)
            quantity = int(quantity)
            if quantity <= 0:
                raise ValueError("Quantity must be greater than zero.")
            quantities[food_id] = quantities.get(food_id, 0) + quantity

        if not quantities:
            raise ValueError("Choose at least one food item.")

        created_at = datetime.now().astimezone().isoformat(timespec="seconds")

        with self._connect() as connection:
            order_cursor = connection.execute(
                """
                INSERT INTO orders
                    (customer_name, cafeteria, destination, priority,
                     status, created_at, total)
                VALUES (?, ?, ?, ?, 'PENDING', ?, 0)
                """,
                (customer_name, cafeteria, destination, priority, created_at),
            )
            order_id = order_cursor.lastrowid
            total = 0.0

            for food_id, quantity in quantities.items():
                item = connection.execute(
                    """
                    SELECT price, is_available
                    FROM food_items
                    WHERE food_id = ?
                    """,
                    (food_id,),
                ).fetchone()

                if item is None:
                    raise ValueError(f"No food item has ID {food_id}.")
                if not item["is_available"]:
                    raise ValueError(f"Food item {food_id} is unavailable.")

                unit_price = float(item["price"])
                total += unit_price * quantity
                connection.execute(
                    """
                    INSERT INTO order_items
                        (order_id, food_id, quantity, unit_price)
                    VALUES (?, ?, ?, ?)
                    """,
                    (order_id, food_id, quantity, unit_price),
                )

            connection.execute(
                "UPDATE orders SET total = ? WHERE order_id = ?",
                (total, order_id),
            )

        return order_id, total

    def update_order_status(self, order_id, status):
        """Update the progress status of an order."""
        status = status.strip().upper()
        if status not in ORDER_STATUSES:
            allowed = ", ".join(sorted(ORDER_STATUSES))
            raise ValueError(f"Status must be one of: {allowed}.")

        with self._connect() as connection:
            cursor = connection.execute(
                "UPDATE orders SET status = ? WHERE order_id = ?",
                (status, order_id),
            )
            if cursor.rowcount == 0:
                raise ValueError(f"No order has ID {order_id}.")

    def list_orders(self):
        """Return saved orders with their item names, quantities, and prices."""
        with self._connect() as connection:
            orders = connection.execute(
                "SELECT * FROM orders ORDER BY order_id DESC"
            ).fetchall()

            result = []
            for order in orders:
                items = connection.execute(
                    """
                    SELECT food_items.name, order_items.quantity,
                           order_items.unit_price
                    FROM order_items
                    JOIN food_items USING (food_id)
                    WHERE order_items.order_id = ?
                    ORDER BY order_items.order_item_id
                    """,
                    (order["order_id"],),
                ).fetchall()

                result.append(
                    {
                        **dict(order),
                        "items": [dict(item) for item in items],
                    }
                )

            return result


def print_menu(database):
    print("\nFood menu:")
    for item in database.list_menu():
        print(
            f"{item['food_id']}. {item['name']} ({item['category']}) "
            f"- Rs. {item['price']:.2f}"
        )


def print_orders(database):
    orders = database.list_orders()
    if not orders:
        print("No saved orders yet.")
        return

    print("\nSaved orders:")
    for order in orders:
        print(
            f"\nOrder #{order['order_id']} | {order['customer_name']} | "
            f"{order['cafeteria']} -> {order['destination']}"
        )
        print(
            f"Status: {order['status']} | Priority: {order['priority']} | "
            f"Total: Rs. {order['total']:.2f}"
        )
        for item in order["items"]:
            line_total = item["quantity"] * item["unit_price"]
            print(
                f"  - {item['name']} x {item['quantity']} "
                f"= Rs. {line_total:.2f}"
            )


def main():
    database = FoodOrderDatabase()
    database.initialize()
    print_menu(database)

    print("\n1. Create a sample order")
    print("2. View saved orders")
    choice = input("Choose 1 or 2: ").strip()

    if choice == "1":
        try:
            customer = input("Customer name: ").strip()
            food_id = int(input("Enter food item ID from the menu: "))
            quantity = int(input("Quantity: "))
            cafeteria = input(
                "Cafeteria ID (CAFE_BLOCK or VIVEKANANDA_CAFE): "
            ).strip()
            destination = input(
                "Destination ID (CHANAKYA, VEDANTA, LIBRARY, HOSTEL, "
                "VISVESVARAYA): "
            ).strip()

            order_id, total = database.create_order(
                customer_name=customer,
                cafeteria=cafeteria,
                destination=destination,
                selections=[(food_id, quantity)],
            )
            print(f"\nSaved order #{order_id}. Total: Rs. {total:.2f}")
            print_orders(database)
        except (ValueError, sqlite3.Error) as error:
            print(f"Could not save the order: {error}")
    elif choice == "2":
        print_orders(database)
    else:
        print("Please enter 1 or 2.")

"""DIT campus food-delivery route simulation.

The coordinates, junctions, distances, and connections below are DEMO DATA.
Replace them with paths and measurements verified on campus before using the
map as a real representation of DIT.

Install the optional visualization dependency with:
    python -m pip install matplotlib
Run with:
    python campus_food_delivery_agent.py
"""

from collections import defaultdict, deque
import heapq
import math
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation



# ---------------------------------------------------------------------------
# 1. Campus map data (DEMO ONLY — replace with surveyed campus data)
# Coordinates are local map positions in metres: (x, y).
# ---------------------------------------------------------------------------

NODE_COORDS = {
    "CAFE_BLOCK": (0, 0),
    "VIVEKANANDA_CAFE": (40, 0),
    "J1": (15, 20),
    "J2": (45, 20),
    "J3": (80, 40),
    "J4": (70, 60),
    "CHANAKYA": (10, 55),
    "VEDANTA": (100, 50),
    "LIBRARY": (100, 80),
    "HOSTEL": (110, 10),
    "VISVESVARAYA": (70, 70),
}

CAFETERIAS = ["CAFE_BLOCK", "VIVEKANANDA_CAFE"]
DESTINATIONS = [
    "CHANAKYA",
    "VEDANTA",
    "LIBRARY",
    "HOSTEL",
    "VISVESVARAYA",
]

# Estimated simulation speed; replace with your robot's measured speed.
ROBOT_SPEED_METRES_PER_MINUTE = 30

# DEMO crowd delays for selected walkways. These are added to the distance / speed
# estimate; replace them with observations if you measure actual campus traffic.
EDGE_CROWD_DELAY_MINUTES = {
    frozenset(("J1", "J3")): 0.8,
    frozenset(("J2", "CHANAKYA")): 1.25,
    frozenset(("J3", "VEDANTA")): 0.2,
}

# Illustrative time spent waiting before a robot is dispatched. This affects
# end-to-end ETA, while priority route selection still uses travel conditions.
PRIORITY_DISPATCH_DELAY_MINUTES = {
    "NORMAL": 1.5,
    "HIGH": 0.5,
    "URGENT": 0.0,
}

# Used only to control how quickly the animation plays, not real robot speed.
ANIMATION_STEPS_PER_METRE = 0.8

campus_graph = defaultdict(dict)
for location_id in NODE_COORDS:
    campus_graph[location_id] = {}


def add_path(place_a, place_b, distance_metres, bidirectional=True):
    """Add a direct walkable path. Set bidirectional=False for one-way paths."""
    if place_a not in NODE_COORDS or place_b not in NODE_COORDS:
        raise ValueError(f"Add coordinates for {place_a!r} and {place_b!r} first.")
    if distance_metres <= 0:
        raise ValueError("Path distance must be greater than zero.")

    campus_graph[place_a][place_b] = float(distance_metres)
    if bidirectional:
        campus_graph[place_b][place_a] = float(distance_metres)


# Demo connections only. Replace these with direct paths verified on campus.
add_path("CAFE_BLOCK", "J1", 30)
add_path("VIVEKANANDA_CAFE", "J2", 24)
add_path("J1", "J2", 33)
add_path("J1", "CHANAKYA", 50)
add_path("J2", "CHANAKYA", 60)
add_path("J1", "J3", 75)
add_path("J2", "J3", 45)
add_path("CHANAKYA", "J3", 75)
add_path("J3", "J4", 25)
add_path("J4", "VISVESVARAYA", 15)
add_path("J4", "VEDANTA", 35)
add_path("J3", "VEDANTA", 25)
add_path("J3", "LIBRARY", 48)
add_path("J3", "HOSTEL", 46)


# ---------------------------------------------------------------------------
# 2. Search and route-cost functions
# ---------------------------------------------------------------------------


def route_distance(graph, route):
    """Return total route distance in metres."""
    return sum(graph[a][b] for a, b in zip(route, route[1:]))


def route_time_minutes(graph, route):
    """Estimate travel time, including the demo delay assigned to each walkway."""
    walking_time = route_distance(graph, route) / ROBOT_SPEED_METRES_PER_MINUTE
    crowd_delay = sum(
        EDGE_CROWD_DELAY_MINUTES.get(frozenset((place_a, place_b)), 0.0)
        for place_a, place_b in zip(route, route[1:])
    )
    return walking_time + crowd_delay


def all_simple_paths(graph, start, goal):
    """List routes from start to goal without visiting a location twice."""
    if start not in graph or goal not in graph:
        return []

    routes = []

    def search(current, path):
        if current == goal:
            routes.append(path.copy())
            return

        for next_location in graph[current]:
            if next_location not in path:
                search(next_location, path + [next_location])

    search(start, [start])
    return routes


def bfs_path(graph, start, goal):
    """Return a path with the fewest edges; BFS ignores distance weights."""
    if start not in graph or goal not in graph:
        return None

    queue = deque([start])
    parent = {start: None}

    while queue:
        current = queue.popleft()
        if current == goal:
            break

        for next_location in graph[current]:
            if next_location not in parent:
                parent[next_location] = current
                queue.append(next_location)

    if goal not in parent:
        return None

    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    return list(reversed(path))


def dfs_visit_order(graph, start):
    """Return DFS visitation order from start."""
    if start not in graph:
        return []

    visited = set()
    order = []

    def visit(current):
        visited.add(current)
        order.append(current)
        for next_location in graph[current]:
            if next_location not in visited:
                visit(next_location)

    visit(start)
    return order


def heuristic_metres(location, goal):
    """Straight-line distance estimate used by A* (coordinates must be metres)."""
    x1, y1 = NODE_COORDS[location]
    x2, y2 = NODE_COORDS[goal]
    return math.hypot(x2 - x1, y2 - y1)


def astar_path(graph, start, goal):
    """Return a lowest-distance path using A*."""
    if start not in graph or goal not in graph:
        return None

    queue = [(heuristic_metres(start, goal), 0.0, start)]
    distance_so_far = {start: 0.0}
    parent = {start: None}

    while queue:
        _, current_distance, current = heapq.heappop(queue)

        if current_distance != distance_so_far.get(current):
            continue

        if current == goal:
            path = []
            location = goal
            while location is not None:
                path.append(location)
                location = parent[location]
            return list(reversed(path))

        for next_location, edge_distance in graph[current].items():
            new_distance = current_distance + edge_distance
            if new_distance < distance_so_far.get(next_location, float("inf")):
                distance_so_far[next_location] = new_distance
                parent[next_location] = current
                priority = new_distance + heuristic_metres(next_location, goal)
                heapq.heappush(queue, (priority, new_distance, next_location))

    return None


def astar_time_path(graph, start, goal):
    """Return the route with the lowest estimated travel time using A*."""
    if start not in graph or goal not in graph:
        return None

    queue = [(heuristic_metres(start, goal) / ROBOT_SPEED_METRES_PER_MINUTE, 0.0, start)]
    time_so_far = {start: 0.0}
    parent = {start: None}

    while queue:
        _, current_time, current = heapq.heappop(queue)
        if current_time != time_so_far.get(current):
            continue

        if current == goal:
            path = []
            location = goal
            while location is not None:
                path.append(location)
                location = parent[location]
            return list(reversed(path))

        for next_location, edge_distance in graph[current].items():
            edge_time = (
                edge_distance / ROBOT_SPEED_METRES_PER_MINUTE
                + EDGE_CROWD_DELAY_MINUTES.get(
                    frozenset((current, next_location)), 0.0
                )
            )
            new_time = current_time + edge_time
            if new_time < time_so_far.get(next_location, float("inf")):
                time_so_far[next_location] = new_time
                parent[next_location] = current
                estimated_total = (
                    new_time
                    + heuristic_metres(next_location, goal)
                    / ROBOT_SPEED_METRES_PER_MINUTE
                )
                heapq.heappush(
                    queue, (estimated_total, new_time, next_location)
                )

    return None


def priority_route(graph, start, goal, priority):
    """Choose a demo route according to the requested delivery priority.

    NORMAL chooses the least-distance A* path. HIGH and URGENT choose the
    lowest estimated travel-time A* path, including configured crowd delays.
    Priority dispatch waiting is reported separately in the end-to-end ETA.
    """
    normalized_priority = (priority or "NORMAL").strip().upper()
    if normalized_priority not in PRIORITY_DISPATCH_DELAY_MINUTES:
        raise ValueError("Priority must be NORMAL, HIGH, or URGENT.")
    if normalized_priority == "NORMAL":
        return astar_path(graph, start, goal)
    return astar_time_path(graph, start, goal)


def estimated_delivery_eta(graph, route, priority):
    """Return travel estimate plus the illustrative priority dispatch wait."""
    if not route:
        return None
    normalized_priority = (priority or "NORMAL").strip().upper()
    return (
        route_time_minutes(graph, route)
        + PRIORITY_DISPATCH_DELAY_MINUTES[normalized_priority]
    )


def print_route(graph, route):
    if not route:
        print("No route found.")
        return

    print(" -> ".join(route))
    print(f"Distance: {route_distance(graph, route):.1f} m")
    print(f"Estimated travel time: {route_time_minutes(graph, route):.2f} min")


# ---------------------------------------------------------------------------
# 3. Live route animation
# ---------------------------------------------------------------------------


def live_agent_view(start, destination, priority="NORMAL"):
    """Animate a small robot along the route chosen for the order priority."""
    route = priority_route(campus_graph, start, destination, priority)
    if route is None:
        print(f"No route found from {start} to {destination}.")
        return

    print(f"{priority} priority selected this route:")
    print_route(campus_graph, route)
    print(
        "Estimated end-to-end ETA including demo dispatch wait: "
        f"{estimated_delivery_eta(campus_graph, route, priority):.2f} min"
    )

    fig, ax = plt.subplots(figsize=(11, 8))
    drawn_edges = set()

    # Draw the complete campus graph in gray.
    for place_a, neighbours in campus_graph.items():
        for place_b, distance in neighbours.items():
            edge_key = tuple(sorted((place_a, place_b)))
            if edge_key in drawn_edges:
                continue
            drawn_edges.add(edge_key)

            x1, y1 = NODE_COORDS[place_a]
            x2, y2 = NODE_COORDS[place_b]
            ax.plot([x1, x2], [y1, y2], color="lightgray", linewidth=2, zorder=1)

            mid_x = (x1 + x2) / 2
            mid_y = (y1 + y2) / 2
            ax.text(mid_x, mid_y, f"{distance:.0f} m", fontsize=8, color="dimgray")

    # Highlight the selected priority route.
    route_x = [NODE_COORDS[place][0] for place in route]
    route_y = [NODE_COORDS[place][1] for place in route]
    ax.plot(
        route_x,
        route_y,
        color="dodgerblue",
        linewidth=5,
        label=f"{priority} chosen route",
        zorder=2,
    )

    # Draw locations. Cafeterias are green, delivery destinations orange.
    for place, (x, y) in NODE_COORDS.items():
        if place in CAFETERIAS:
            colour = "seagreen"
        elif place in DESTINATIONS:
            colour = "orange"
        else:
            colour = "lightsteelblue"

        ax.scatter(x, y, color=colour, s=150, edgecolor="black", zorder=3)
        ax.text(x, y + 2.5, place, ha="center", fontsize=9, zorder=4)

    # Create intermediate positions for the moving robot icon.
    frames = []
    for place_a, place_b in zip(route, route[1:]):
        x1, y1 = NODE_COORDS[place_a]
        x2, y2 = NODE_COORDS[place_b]
        distance = campus_graph[place_a][place_b]
        steps = max(10, int(distance * ANIMATION_STEPS_PER_METRE))

        for step in range(steps):
            progress = step / steps
            x = x1 + (x2 - x1) * progress
            y = y1 + (y2 - y1) * progress
            frames.append((x, y, place_a, place_b, progress))

    final_x, final_y = NODE_COORDS[destination]
    frames.append((final_x, final_y, destination, destination, 1.0))

    robot_body, = ax.plot(
        [], [], marker="s", markersize=11,
        markerfacecolor="#168a85", markeredgecolor="white",
        markeredgewidth=1.5, linestyle="", zorder=6,
    )
    robot_head, = ax.plot(
        [], [], marker="s", markersize=7,
        markerfacecolor="#dce6f1", markeredgecolor="#344054",
        linestyle="", zorder=7,
    )
    robot_wheels, = ax.plot(
        [], [], marker="o", markersize=4,
        markerfacecolor="#263238", markeredgecolor="#263238",
        linestyle="", zorder=5,
    )
    robot_antenna, = ax.plot([], [], color="#344054", linewidth=2, zorder=7)
    trail, = ax.plot([], [], color="#168a85", linewidth=2, alpha=0.65, zorder=4)
    status = ax.text(
        0.02, 0.98, "",
        transform=ax.transAxes,
        va="top",
        bbox={"facecolor": "white", "alpha": 0.9},
    )

    def update(frame_number):
        x, y, from_place, to_place, progress = frames[frame_number]
        robot_body.set_data([x], [y])
        robot_head.set_data([x], [y + 1.25])
        robot_wheels.set_data([x - 0.75, x + 0.75], [y - 0.65, y - 0.65])
        robot_antenna.set_data([x, x], [y + 1.7, y + 2.5])
        trail.set_data(
            [frame[0] for frame in frames[:frame_number + 1]],
            [frame[1] for frame in frames[:frame_number + 1]],
        )

        if from_place == to_place:
            movement = f"Arrived at {destination}"
        else:
            movement = f"Moving: {from_place} -> {to_place} ({progress:.0%})"

        status.set_text(
            f"{priority} route | ETA: "
            f"{estimated_delivery_eta(campus_graph, route, priority):.2f} min\n"
            f"{' -> '.join(route)}\n{movement}"
        )
        return robot_body, robot_head, robot_wheels, robot_antenna, trail, status

    ax.set_title(f"Campus Delivery Simulation: {start} to {destination} ({priority})")
    ax.set_xlabel("Local map X (metres)")
    ax.set_ylabel("Local map Y (metres)")
    ax.grid(True)
    ax.legend(loc="lower right")
    ax.set_aspect("equal", adjustable="datalim")

    # Keep a reference to the animation until the window closes.
    animation = FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=100,
        repeat=False,
        blit=False,
    )

    # Colab/Jupyter needs HTML output to render a Matplotlib animation.
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        from IPython.display import HTML, display

        display(HTML(animation.to_jshtml()))
        plt.close(fig)
    else:
        plt.show()

    return animation


# ---------------------------------------------------------------------------
# 4. Command-line entry point
# ---------------------------------------------------------------------------


def main():
    print("DIT Campus Food Delivery — Orders and Route Simulation")
    print("NOTE: The map connections and measurements are demo data.\n")

    database = FoodOrderDatabase()
    database.initialize()

    print("1. Place a food order and simulate its delivery")
    print("2. View saved orders")
    action = input("Choose 1 or 2: ").strip()

    if action == "2":
        print_orders(database)
        return
    if action != "1":
        print("Please enter 1 or 2.")
        return

    print_menu(database)
    customer = input("\nCustomer name: ").strip()

    try:
        selections = []
        while True:
            food_id = int(input("Enter a food item ID: "))
            quantity = int(input("Quantity: "))
            selections.append((food_id, quantity))
            add_another = input("Add another food item? (y/n): ").strip().lower()
            if add_another != "y":
                break

        print("\nCafeteria IDs:", ", ".join(CAFETERIAS))
        print("Destination IDs:", ", ".join(DESTINATIONS))
        start = input("Enter the cafeteria ID: ").strip().upper()
        destination = input("Enter the destination ID: ").strip().upper()
        priority = input("Order priority (NORMAL, HIGH, URGENT): ").strip().upper()
        priority = priority or "NORMAL"

        if start not in CAFETERIAS:
            print("Unknown cafeteria ID. Use one of the IDs shown above.")
            return
        if destination not in DESTINATIONS:
            print("Unknown destination ID. Use one of the IDs shown above.")
            return

        chosen_route = priority_route(campus_graph, start, destination, priority)
        if chosen_route is None:
            print("No route is connected in the current demo map.")
            return

        order_id, total = database.create_order(
            customer_name=customer,
            cafeteria=start,
            destination=destination,
            selections=selections,
            priority=priority,
        )
        database.update_order_status(order_id, "PREPARING")
        print(f"\nOrder #{order_id} saved. Total: Rs. {total:.2f}")

    except (ValueError, TypeError) as error:
        print(f"Could not create the order: {error}")
        return

    routes = all_simple_paths(campus_graph, start, destination)
    routes.sort(
        key=lambda path: (
            route_time_minutes(campus_graph, path),
            route_distance(campus_graph, path),
        )
    )

    print(f"\nAll simple routes from {start} to {destination}:")
    for index, candidate_route in enumerate(routes, start=1):
        print(f"\nRoute {index}:")
        print_route(campus_graph, candidate_route)

    print("\nBFS route (fewest edges; ignores distance):")
    print_route(campus_graph, bfs_path(campus_graph, start, destination))

    print("\nDFS visit order from the cafeteria:")
    print(" -> ".join(dfs_visit_order(campus_graph, start)))

    print("\nPriority-selected route:")
    print_route(campus_graph, chosen_route)
    print(
        "Estimated end-to-end ETA including demo dispatch wait: "
        f"{estimated_delivery_eta(campus_graph, chosen_route, priority):.2f} min"
    )
    print("\nOpening route animation. Close the plot window to finish.")
    database.update_order_status(order_id, "OUT_FOR_DELIVERY")
    live_agent_view(start, destination, priority=priority)
    database.update_order_status(order_id, "DELIVERED")
    print_orders(database)


In [ ]:
"""Interactive Colab dashboard cell for the campus food delivery notebook."""

from html import escape

try:
    import ipywidgets as widgets
    import plotly.graph_objects as go
    from IPython.display import HTML, clear_output, display
except ImportError:
    widgets = None
    go = None
    HTML = clear_output = display = None

try:
    from google.colab import output as colab_output

    colab_output.enable_custom_widget_manager()
except (ImportError, AttributeError):
    pass


# Prospectus-inspired DIT facade forms: brick-red walls, pale floor bands,
# repeated windows, flat roofs, and landscaped surroundings. Geometry is still
# illustrative and must be replaced with measured building data for accuracy.
BUILDING_MODEL = {
    "CAFE_BLOCK": {
        "label": "Main Cafeteria",
        "center": (-5, -9), "width": 12, "depth": 10,
        "floors": 2, "front": "north", "ground_open": False,
        "brick": "#9d4b38", "accent": "#eee7da",
    },
    "VIVEKANANDA_CAFE": {
        "label": "Vivekananda Cafe",
        "center": (47, -8), "width": 12, "depth": 10,
        "floors": 3, "front": "north", "ground_open": True,
        "brick": "#a14e3c", "accent": "#eee8dd",
    },
    "CHANAKYA": {
        "label": "Chanakya Building",
        "center": (3, 64), "width": 14, "depth": 12,
        "floors": 4, "front": "south", "ground_open": True,
        "brick": "#984633", "accent": "#f1e9dd",
    },
    "VEDANTA": {
        "label": "Vedanta Building",
        "center": (111, 53), "width": 14, "depth": 12,
        "floors": 5, "front": "west", "ground_open": True,
        "brick": "#a24d39", "accent": "#f2ece2",
    },
    "LIBRARY": {
        "label": "Library",
        "center": (111, 88), "width": 16, "depth": 12,
        "floors": 4, "front": "west", "ground_open": True,
        "brick": "#a85a45", "accent": "#f1eade",
    },
    "HOSTEL": {
        "label": "Hostel",
        "center": (120, 4), "width": 14, "depth": 12,
        "floors": 5, "front": "west", "ground_open": False,
        "brick": "#9a4935", "accent": "#eee8df",
    },
    "VISVESVARAYA": {
        "label": "Visvesvaraya Building",
        "center": (82, 72), "width": 14, "depth": 12,
        "floors": 4, "front": "west", "ground_open": True,
        "brick": "#9f4b38", "accent": "#f0e9df",
    },
}


def launch_gui():
    """Create the order-entry, route-preview, and order-history dashboard."""
    if widgets is None or go is None:
        raise RuntimeError(
            "Install matplotlib, ipywidgets, and plotly before starting the dashboard."
        )

    database = FoodOrderDatabase()
    database.initialize()
    cart = {}
    active_order_id = {"value": None}
    menu_items = database.list_menu()
    menu_by_id = {item["food_id"]: item for item in menu_items}

    header = widgets.HTML(
        """
        <div style="padding:22px 24px;border-radius:16px;color:white;
                    background:linear-gradient(120deg,#123b5d,#168a85);
                    font-family:Arial,sans-serif;">
          <div style="font-size:13px;opacity:.85;letter-spacing:1px;">
            DIT CAMPUS PROJECT
          </div>
          <div style="font-size:28px;font-weight:700;margin-top:5px;">
            Campus Food Delivery
          </div>
          <div style="font-size:14px;margin-top:7px;opacity:.92;">
            Build an order, compare priority routes and travel times, and explore a DIT-inspired campus model in 3D.
          </div>
        </div>
        """,
        layout=widgets.Layout(width="100%"),
    )

    customer = widgets.Text(
        description="Customer",
        placeholder="Enter student or department name",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="100%"),
    )
    cafeteria = widgets.Dropdown(
        options=CAFETERIAS,
        value=CAFETERIAS[0],
        description="Pickup cafeteria",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="33%"),
    )
    destination = widgets.Dropdown(
        options=DESTINATIONS,
        value="VISVESVARAYA",
        description="Delivery location",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="37%"),
    )
    priority = widgets.Dropdown(
        options=["NORMAL", "HIGH", "URGENT"],
        value="NORMAL",
        description="Priority",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="30%"),
    )

    menu_options = [
        (f"{item['name']} — ₹{item['price']:.2f}", item["food_id"])
        for item in menu_items
    ]
    food = widgets.Dropdown(
        options=menu_options,
        description="Food item",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="70%"),
    )
    quantity = widgets.BoundedIntText(
        value=1,
        min=1,
        max=99,
        description="Quantity",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="30%"),
    )

    add_button = widgets.Button(
        description="Add to cart",
        icon="plus",
        button_style="info",
        tooltip="Add the selected item and quantity",
        layout=widgets.Layout(width="160px", height="38px"),
    )
    remove_button = widgets.Button(
        description="Remove selected",
        icon="trash",
        button_style="warning",
        tooltip="Remove the selected item from the cart",
        layout=widgets.Layout(width="180px", height="38px"),
    )
    preview_button = widgets.Button(
        description="Preview priority routes",
        icon="map",
        button_style="",
        tooltip="Compare the selected priority route with alternatives and ETAs",
        layout=widgets.Layout(width="220px", height="40px"),
    )
    place_button = widgets.Button(
        description="Place order & show 3D route",
        icon="check",
        button_style="success",
        tooltip="Save the order and display an animated 3D delivery route",
        layout=widgets.Layout(width="260px", height="42px"),
    )
    mark_delivered_button = widgets.Button(
        description="Mark delivered",
        icon="check-circle",
        button_style="success",
        tooltip="Mark the active simulated delivery as delivered",
        disabled=True,
        layout=widgets.Layout(width="190px", height="40px"),
    )
    history_button = widgets.Button(
        description="Show order history",
        icon="list",
        button_style="",
        layout=widgets.Layout(width="210px", height="40px"),
    )

    cart_output = widgets.Output()
    message_output = widgets.Output()
    route_output = widgets.Output(
        layout=widgets.Layout(width="100%", overflow="auto")
    )
    history_output = widgets.Output(
        layout=widgets.Layout(width="100%", overflow="auto")
    )

    def refresh_cart():
        with cart_output:
            clear_output(wait=True)
            if not cart:
                display(
                    HTML(
                        "<div style='padding:12px;color:#667085;'>"
                        "Your cart is empty. Choose an item and click "
                        "<b>Add to cart</b>.</div>"
                    )
                )
                return

            rows = []
            total = 0.0
            for food_id, item_quantity in cart.items():
                item = menu_by_id[food_id]
                line_total = item["price"] * item_quantity
                total += line_total
                rows.append(
                    "<tr>"
                    f"<td>{escape(item['name'])}</td>"
                    f"<td style='text-align:center'>{item_quantity}</td>"
                    f"<td style='text-align:right'>₹{line_total:.2f}</td>"
                    "</tr>"
                )
            table = (
                "<table style='border-collapse:collapse;width:100%;'>"
                "<thead><tr style='background:#f2f6f8;'>"
                "<th style='text-align:left;padding:8px'>Item</th>"
                "<th style='padding:8px'>Qty</th>"
                "<th style='text-align:right;padding:8px'>Amount</th>"
                "</tr></thead><tbody>"
                + "".join(rows)
                + "</tbody></table>"
                f"<div style='text-align:right;padding:10px;font-weight:700'>"
                f"Cart total: ₹{total:.2f}</div>"
            )
            display(HTML(table))

    def create_route_plan(start, goal, selected_priority):
        """Return the selected route, priority scenarios, and fastest alternatives."""
        scenarios = []
        for scenario_priority in ("NORMAL", "HIGH", "URGENT"):
            scenario_route = priority_route(
                campus_graph, start, goal, scenario_priority
            )
            if scenario_route is None:
                continue
            scenarios.append(
                {
                    "priority": scenario_priority,
                    "route": scenario_route,
                    "distance": route_distance(campus_graph, scenario_route),
                    "travel_time": route_time_minutes(campus_graph, scenario_route),
                    "dispatch_wait": PRIORITY_DISPATCH_DELAY_MINUTES[scenario_priority],
                    "eta": estimated_delivery_eta(
                        campus_graph, scenario_route, scenario_priority
                    ),
                }
            )

        selected_scenario = next(
            (item for item in scenarios if item["priority"] == selected_priority),
            None,
        )
        if selected_scenario is None:
            return None, scenarios, []

        alternatives = all_simple_paths(campus_graph, start, goal)
        alternatives.sort(
            key=lambda path: (
                route_time_minutes(campus_graph, path),
                route_distance(campus_graph, path),
            )
        )
        return selected_scenario["route"], scenarios, alternatives[:5]

    def route_report_html(start, goal, selected_priority, scenarios, alternatives, heading):
        """Build priority comparisons and ETA estimates for the dashboard."""
        cell_style = (
            "padding:8px 9px;color:#1d2939 !important;"
            "border-bottom:1px solid #d0d5dd;"
        )
        number_cell_style = cell_style + "text-align:right;white-space:nowrap;"
        header_style = (
            "padding:8px 9px;background:#e4eaf1;"
            "color:#344054 !important;text-align:left;"
            "border-bottom:2px solid #b8c3d1;white-space:nowrap;"
        )
        number_header_style = header_style + "text-align:right;"
        table_style = (
            "border-collapse:collapse;width:100%;font-size:13px;"
            "color:#1d2939 !important;background:#ffffff;"
        )
        scenario_rows = []
        for item in scenarios:
            selected_style = (
                "background:#dcfce7;font-weight:700;color:#14532d !important;"
                if item["priority"] == selected_priority else ""
            )
            scenario_rows.append(
                f"<tr style='{selected_style}'>"
                f"<td style='{cell_style}{selected_style}'>{item['priority']}</td>"
                f"<td style='{cell_style}{selected_style}'>{escape(' → '.join(item['route']))}</td>"
                f"<td style='{number_cell_style}{selected_style}'>{item['distance']:.0f} m</td>"
                f"<td style='{number_cell_style}{selected_style}'>{item['travel_time']:.2f} min</td>"
                f"<td style='{number_cell_style}{selected_style}'>{item['dispatch_wait']:.2f} min</td>"
                f"<td style='{number_cell_style}{selected_style}'>{item['eta']:.2f} min</td></tr>"
            )

        alternative_rows = []
        for index, candidate in enumerate(alternatives, start=1):
            candidate_time = route_time_minutes(campus_graph, candidate)
            candidate_distance = route_distance(campus_graph, candidate)
            candidate_eta = candidate_time + PRIORITY_DISPATCH_DELAY_MINUTES[selected_priority]
            alternative_rows.append(
                f"<tr><td style='{cell_style}'>{index}</td>"
                f"<td style='{cell_style}'>{escape(' → '.join(candidate))}</td>"
                f"<td style='{number_cell_style}'>{candidate_distance:.0f} m</td>"
                f"<td style='{number_cell_style}'>{candidate_time:.2f} min</td>"
                f"<td style='{number_cell_style}'>{candidate_eta:.2f} min</td></tr>"
            )

        return (
            "<div style='padding:12px;background:#f8fafc;border:1px solid #d0d5dd;"
            "border-radius:10px;margin-bottom:10px;color:#1d2939 !important;"
            "font-family:Arial,sans-serif'>"
            f"<h4 style='margin:0 0 8px;color:#102a43 !important'>{escape(heading)}: "
            f"{escape(start)} → {escape(goal)}</h4>"
            "<div style='font-size:13px;color:#475467 !important;margin-bottom:8px'>"
            "NORMAL selects the shortest-distance path. HIGH and URGENT select "
            "the fastest estimated path using demo crowd-delay values. Priority "
            "changes simulated dispatch wait, not robot speed.</div>"
            f"<div style='overflow-x:auto'><table style='{table_style}'><thead><tr>"
            f"<th style='{header_style}'>Priority</th>"
            f"<th style='{header_style}'>Chosen path</th>"
            f"<th style='{number_header_style}'>Distance</th>"
            f"<th style='{number_header_style}'>Travel time</th>"
            f"<th style='{number_header_style}'>Dispatch wait</th>"
            f"<th style='{number_header_style}'>ETA</th>"
            "</tr></thead><tbody>" + "".join(scenario_rows) + "</tbody></table></div>"
            "<h4 style='margin:12px 0 6px;color:#102a43 !important'>"
            "Alternative paths (ranked by estimated travel time)</h4>"
            f"<div style='overflow-x:auto'><table style='{table_style}'><thead><tr>"
            f"<th style='{header_style}'>#</th>"
            f"<th style='{header_style}'>Path</th>"
            f"<th style='{number_header_style}'>Distance</th>"
            f"<th style='{number_header_style}'>Travel time</th>"
            f"<th style='{number_header_style}'>ETA with {escape(selected_priority)} wait</th>"
            "</tr></thead><tbody>" + "".join(alternative_rows) + "</tbody></table></div>"
            "<div style='font-size:11px;color:#475467 !important;margin-top:8px'>"
            "Travel time uses the demo speed (30 m/min) plus estimated walkway delays. "
            "ETA adds the illustrative priority dispatch wait. Edit the values in "
            "EDGE_CROWD_DELAY_MINUTES and PRIORITY_DISPATCH_DELAY_MINUTES to calibrate it.</div></div>"
        )

    def draw_campus_map(route=None, animate=False, comparison_routes=None,
                        selected_priority="NORMAL"):
        """Render the high-resolution campus model and priority route comparison."""
        fig = go.Figure()

        # A light ground plane makes the buildings and routes easier to read.
        fig.add_trace(
            go.Mesh3d(
                x=[-20, 140, 140, -20],
                y=[-25, -25, 105, 105],
                z=[0, 0, 0, 0],
                i=[0, 0], j=[1, 2], k=[2, 3],
                color="#e7f0e5", opacity=0.55,
                hoverinfo="skip", showscale=False, name="Campus ground",
            )
        )

        # Draw all walkable graph edges at ground level.
        road_x, road_y, road_z, road_labels = [], [], [], []
        distance_x, distance_y, distance_z, distance_labels = [], [], [], []
        drawn_edges = set()
        for place_a, neighbours in campus_graph.items():
            for place_b, distance in neighbours.items():
                edge_key = tuple(sorted((place_a, place_b)))
                if edge_key in drawn_edges:
                    continue
                drawn_edges.add(edge_key)
                x1, y1 = NODE_COORDS[place_a]
                x2, y2 = NODE_COORDS[place_b]
                road_x.extend([x1, x2, None])
                road_y.extend([y1, y2, None])
                road_z.extend([0.35, 0.35, None])
                distance_x.append((x1 + x2) / 2)
                distance_y.append((y1 + y2) / 2)
                distance_z.append(0.8)
                distance_labels.append(f"{distance:.0f} m")

        fig.add_trace(
            go.Scatter3d(
                x=road_x, y=road_y, z=road_z,
                mode="lines", name="Campus paths",
                line={"color": "#aab4c0", "width": 5},
                hoverinfo="skip",
            )
        )
        fig.add_trace(
            go.Scatter3d(
                x=distance_x, y=distance_y, z=distance_z,
                mode="text", text=distance_labels,
                textfont={"size": 9, "color": "#596579"},
                showlegend=False, hoverinfo="skip",
            )
        )

        # Build detailed, low-rise DIT-inspired academic facades. Each building
        # combines brick-red walls, pale slab bands, repeated glass windows,
        # concrete pilotis, a small entry canopy, and a flat parapet roof.
        floor_height = 3.15
        for node_id, building in BUILDING_MODEL.items():
            center_x, center_y = building["center"]
            width, depth = building["width"], building["depth"]
            floors = building["floors"]
            height = floors * floor_height
            x1, x2 = center_x - width / 2, center_x + width / 2
            y1, y2 = center_y - depth / 2, center_y + depth / 2
            front = building["front"]
            front_y = y1 if front == "south" else (y2 if front == "north" else None)
            front_x = x1 if front == "west" else (x2 if front == "east" else None)

            vx, vy, vz = [], [], []
            face_i, face_j, face_k, face_colors = [], [], [], []

            def append_box(cx, cy, z0, box_w, box_d, box_h, color):
                base = len(vx)
                bx1, bx2 = cx - box_w / 2, cx + box_w / 2
                by1, by2 = cy - box_d / 2, cy + box_d / 2
                vx.extend([bx1, bx2, bx2, bx1, bx1, bx2, bx2, bx1])
                vy.extend([by1, by1, by2, by2, by1, by1, by2, by2])
                vz.extend([z0, z0, z0, z0, z0 + box_h, z0 + box_h, z0 + box_h, z0 + box_h])
                local_i = [0, 0, 4, 4, 0, 0, 1, 1, 2, 2, 3, 3]
                local_j = [2, 3, 5, 6, 1, 5, 2, 6, 3, 7, 0, 4]
                local_k = [1, 2, 6, 7, 5, 4, 6, 5, 7, 6, 4, 7]
                face_i.extend(base + value for value in local_i)
                face_j.extend(base + value for value in local_j)
                face_k.extend(base + value for value in local_k)
                face_colors.extend([color] * 12)

            # Brick body and concrete floor slabs.
            open_ground = building["ground_open"]
            body_base = floor_height if open_ground else 0.3
            append_box(
                center_x, center_y, body_base, width, depth,
                max(0.5, height - body_base), building["brick"],
            )
            append_box(
                center_x, center_y, 0, width + 0.8, depth + 0.8, 0.28,
                "#c9c4ba",
            )
            for floor_number in range(1, floors + 1):
                z_floor = floor_number * floor_height - 0.12
                append_box(
                    center_x, center_y, z_floor, width + 0.55,
                    depth + 0.55, 0.24, building["accent"],
                )

            # Open, column-supported entrance level and a shaded portico.
            if open_ground:
                column_count = max(4, int(width / 2.6))
                for column_index in range(column_count):
                    column_x = x1 + 0.45 + column_index * (width - 0.9) / (column_count - 1)
                    append_box(column_x, y1 + 0.35, 0, 0.34, 0.48, floor_height, "#e7e0d5")
                    append_box(column_x, y2 - 0.35, 0, 0.34, 0.48, floor_height, "#e7e0d5")
                side_column_count = max(2, int(depth / 3.6))
                for column_index in range(1, side_column_count):
                    column_y = y1 + column_index * depth / side_column_count
                    append_box(x1 + 0.28, column_y, 0, 0.42, 0.34, floor_height, "#e7e0d5")
                    append_box(x2 - 0.28, column_y, 0, 0.42, 0.34, floor_height, "#e7e0d5")

            if front_y is not None:
                canopy_y = front_y - 1.15 if front == "south" else front_y + 1.15
                append_box(center_x, canopy_y, 2.65, min(width * 0.62, 7.0), 2.5, 0.28, "#eee8dd")
                for side_sign in (-1, 1):
                    append_box(center_x + side_sign * 2.0, canopy_y, 0.25, 0.28, 0.3, 2.4, "#e4ded3")
            else:
                canopy_x = front_x - 1.15 if front == "west" else front_x + 1.15
                append_box(canopy_x, center_y, 2.65, 2.5, min(depth * 0.62, 7.0), 0.28, "#eee8dd")
                for side_sign in (-1, 1):
                    append_box(canopy_x, center_y + side_sign * 2.0, 0.25, 0.3, 0.28, 2.4, "#e4ded3")

            # Subtle vertical pilasters, roof edge, parapets, and rooftop plant.
            bay_count = max(3, int(width / 2.6))
            if front_y is not None:
                for bay_index in range(bay_count + 1):
                    pier_x = x1 + bay_index * width / bay_count
                    append_box(pier_x, front_y, body_base, 0.16, 0.16, height - body_base, "#d9c9b4")
            else:
                for bay_index in range(bay_count + 1):
                    pier_y = y1 + bay_index * depth / bay_count
                    append_box(front_x, pier_y, body_base, 0.16, 0.16, height - body_base, "#d9c9b4")
            append_box(center_x, center_y, height, width + 0.5, depth + 0.5, 0.24, "#e8e1d6")
            append_box(center_x, y1 + 0.12, height + 0.24, width + 0.45, 0.24, 0.72, "#e9e3d8")
            append_box(center_x, y2 - 0.12, height + 0.24, width + 0.45, 0.24, 0.72, "#e9e3d8")
            append_box(x1 + 0.12, center_y, height + 0.24, 0.24, depth + 0.45, 0.72, "#e9e3d8")
            append_box(x2 - 0.12, center_y, height + 0.24, 0.24, depth + 0.45, 0.72, "#e9e3d8")
            append_box(center_x + width * 0.22, center_y + depth * 0.12, height + 0.24, 1.8, 1.4, 0.85, "#aaa9a5")

            fig.add_trace(
                go.Mesh3d(
                    x=vx, y=vy, z=vz,
                    i=face_i, j=face_j, k=face_k,
                    facecolor=face_colors, flatshading=True,
                    showscale=False, name=building["label"], showlegend=False,
                    hovertemplate=(
                        f"{building['label']}<br>"
                        f"{floors} illustrative floors<extra></extra>"
                    ),
                )
            )

            # Repeated blue-grey windows on the front and adjacent side.
            glass_x, glass_y, glass_z = [], [], []
            glass_i, glass_j, glass_k = [], [], []
            glass_colors = []
            frame_x, frame_y, frame_z = [], [], []

            def add_window(orientation, lateral, z_bottom, window_w, window_h, color="#7896a6"):
                base = len(glass_x)
                if orientation == "south":
                    x0, x3 = lateral - window_w / 2, lateral + window_w / 2
                    y_value = y1 - 0.09
                    points = [(x0, y_value, z_bottom), (x3, y_value, z_bottom),
                              (x3, y_value, z_bottom + window_h), (x0, y_value, z_bottom + window_h)]
                elif orientation == "north":
                    x0, x3 = lateral - window_w / 2, lateral + window_w / 2
                    y_value = y2 + 0.09
                    points = [(x0, y_value, z_bottom), (x3, y_value, z_bottom),
                              (x3, y_value, z_bottom + window_h), (x0, y_value, z_bottom + window_h)]
                elif orientation == "west":
                    y0, y3 = lateral - window_w / 2, lateral + window_w / 2
                    x_value = x1 - 0.09
                    points = [(x_value, y0, z_bottom), (x_value, y3, z_bottom),
                              (x_value, y3, z_bottom + window_h), (x_value, y0, z_bottom + window_h)]
                else:
                    y0, y3 = lateral - window_w / 2, lateral + window_w / 2
                    x_value = x2 + 0.09
                    points = [(x_value, y0, z_bottom), (x_value, y3, z_bottom),
                              (x_value, y3, z_bottom + window_h), (x_value, y0, z_bottom + window_h)]
                for point_x, point_y, point_z in points:
                    glass_x.append(point_x)
                    glass_y.append(point_y)
                    glass_z.append(point_z)
                glass_i.extend([base, base])
                glass_j.extend([base + 1, base + 2])
                glass_k.extend([base + 2, base + 3])
                glass_colors.extend([color, color])

                edge_indices = [(0, 1), (1, 2), (2, 3), (3, 0)]
                for first, second in edge_indices:
                    frame_x.extend([points[first][0], points[second][0], None])
                    frame_y.extend([points[first][1], points[second][1], None])
                    frame_z.extend([points[first][2], points[second][2], None])
                if orientation in ("north", "south"):
                    frame_x.extend([lateral, lateral, None,
                                    lateral - window_w / 2, lateral + window_w / 2, None])
                    frame_y.extend([points[0][1], points[2][1], None,
                                    points[0][1], points[1][1], None])
                else:
                    frame_x.extend([points[0][0], points[2][0], None,
                                    points[0][0], points[1][0], None])
                    frame_y.extend([lateral, lateral, None,
                                    lateral - window_w / 2, lateral + window_w / 2, None])
                frame_z.extend([z_bottom, z_bottom + window_h, None,
                                z_bottom + window_h / 2, z_bottom + window_h / 2, None])

            front_side = front
            side_side = "east" if front in ("north", "south") else "north"
            visible_floors = range(1, floors) if open_ground else range(floors)
            for floor_number in visible_floors:
                z_bottom = floor_number * floor_height + 0.78
                window_height = 1.42
                front_span = width if front in ("north", "south") else depth
                front_start = x1 if front in ("north", "south") else y1
                front_bays = max(3, int(front_span / 2.5))
                front_window_w = min(1.45, front_span / (front_bays + 0.6))
                for bay_index in range(front_bays):
                    lateral = front_start + (bay_index + 0.5) * front_span / front_bays
                    add_window(front_side, lateral, z_bottom, front_window_w, window_height)

                side_span = depth if side_side in ("east", "west") else width
                side_start = y1 if side_side in ("east", "west") else x1
                side_bays = max(2, int(side_span / 3.0))
                side_window_w = min(1.3, side_span / (side_bays + 0.7))
                for bay_index in range(side_bays):
                    lateral = side_start + (bay_index + 0.5) * side_span / side_bays
                    add_window(side_side, lateral, z_bottom, side_window_w, window_height)

            # Main entrance glazing below the portico.
            if front in ("north", "south"):
                add_window(front, center_x, 0.45, min(2.2, width * 0.24), 2.05, color="#476777")
            else:
                add_window(front, center_y, 0.45, min(2.2, depth * 0.24), 2.05, color="#476777")

            if glass_x:
                fig.add_trace(
                    go.Mesh3d(
                        x=glass_x, y=glass_y, z=glass_z,
                        i=glass_i, j=glass_j, k=glass_k,
                        facecolor=glass_colors, opacity=0.98,
                        flatshading=True, showscale=False,
                        name=f"{building['label']} windows", showlegend=False,
                        hoverinfo="skip",
                    )
                )
                fig.add_trace(
                    go.Scatter3d(
                        x=frame_x, y=frame_y, z=frame_z,
                        mode="lines", line={"color": "#e8e2d8", "width": 3},
                        showlegend=False, hoverinfo="skip",
                    )
                )

            fig.add_trace(
                go.Scatter3d(
                    x=[center_x], y=[center_y], z=[height + 2.0],
                    mode="text", text=[building["label"]],
                    textfont={"size": 11, "color": "#243247"},
                    showlegend=False, hoverinfo="skip",
                )
            )

        # Trees and lawn points echo the prospectus's green, foothill campus.
        tree_positions = [
            (22, 42), (36, 47), (57, 35), (90, 17), (95, 68),
            (52, 82), (28, -10), (8, 82), (122, 32), (57, -5),
        ]
        trunk_x, trunk_y, trunk_z = [], [], []
        for tree_x, tree_y in tree_positions:
            trunk_x.extend([tree_x, tree_x, None])
            trunk_y.extend([tree_y, tree_y, None])
            trunk_z.extend([0.2, 2.2, None])
        fig.add_trace(
            go.Scatter3d(
                x=trunk_x, y=trunk_y, z=trunk_z,
                mode="lines", line={"color": "#795548", "width": 5},
                showlegend=False, hoverinfo="skip",
            )
        )
        fig.add_trace(
            go.Scatter3d(
                x=[x for x, _ in tree_positions],
                y=[y for _, y in tree_positions],
                z=[2.2] * len(tree_positions),
                mode="markers",
                marker={"size": 14, "color": "#4f8655", "opacity": 0.92},
                name="Campus trees", hoverinfo="skip",
            )
        )

        # Junction nodes stay at street level. Building locations use their
        # separate cuboid and label above.
        junction_ids = [place for place in NODE_COORDS if place not in BUILDING_MODEL]
        fig.add_trace(
            go.Scatter3d(
                x=[NODE_COORDS[place][0] for place in junction_ids],
                y=[NODE_COORDS[place][1] for place in junction_ids],
                z=[1.0] * len(junction_ids),
                mode="markers+text",
                text=junction_ids,
                textposition="top center",
                marker={"size": 5, "color": "#607d9a", "line": {"color": "white", "width": 1}},
                name="Path junctions",
                hovertemplate="Junction: %{text}<extra></extra>",
            )
        )

        if route:
            route_groups = {}
            for scenario in comparison_routes or []:
                scenario_route = tuple(scenario["route"])
                if scenario_route != tuple(route):
                    route_groups.setdefault(scenario_route, []).append(scenario)
            route_colors = {
                "NORMAL": "#3976c5",
                "HIGH": "#d58b15",
                "URGENT": "#8b5cc7",
            }
            for scenario_route, grouped_scenarios in route_groups.items():
                scenario_priorities = [item["priority"] for item in grouped_scenarios]
                scenario_color = route_colors[scenario_priorities[0]]
                scenario_time = grouped_scenarios[0]["travel_time"]
                fig.add_trace(
                    go.Scatter3d(
                        x=[NODE_COORDS[place][0] for place in scenario_route],
                        y=[NODE_COORDS[place][1] for place in scenario_route],
                        z=[0.72] * len(scenario_route),
                        mode="lines+markers",
                        name=(f"{' / '.join(scenario_priorities)} choice "
                              f"({scenario_time:.2f} min travel)"),
                        line={"color": scenario_color, "width": 6, "dash": "dash"},
                        marker={"size": 3, "color": scenario_color},
                        hovertemplate=(
                            f"{' / '.join(scenario_priorities)} priority path<br>"
                            f"{' → '.join(scenario_route)}<br>"
                            f"Travel estimate: {scenario_time:.2f} min<extra></extra>"
                        ),
                    )
                )

            route_x = [NODE_COORDS[place][0] for place in route]
            route_y = [NODE_COORDS[place][1] for place in route]
            selected_distance = route_distance(campus_graph, route)
            selected_travel_time = route_time_minutes(campus_graph, route)
            fig.add_trace(
                go.Scatter3d(
                    x=route_x, y=route_y, z=[0.9] * len(route),
                    mode="lines+markers",
                    name=f"{selected_priority} selected route (solid)",
                    line={"color": "#168a85", "width": 10},
                    marker={"size": 4, "color": "#168a85"},
                    hovertemplate=(
                        f"{selected_priority} selected route<br>"
                        f"{' → '.join(route)}<br>"
                        f"Distance: {selected_distance:.0f} m<br>"
                        f"Travel estimate: {selected_travel_time:.2f} min<extra></extra>"
                    ),
                )
            )

            if animate:
                # Interpolate a modest number of frames along each graph edge.
                positions = []
                for place_a, place_b in zip(route, route[1:]):
                    x1, y1 = NODE_COORDS[place_a]
                    x2, y2 = NODE_COORDS[place_b]
                    for step in range(6):
                        fraction = step / 6
                        positions.append((
                            x1 + (x2 - x1) * fraction,
                            y1 + (y2 - y1) * fraction,
                        ))
                positions.append(NODE_COORDS[route[-1]])
                initial_x, initial_y = positions[0]
                robot_trace_indices = []
                robot_trace_indices.append(len(fig.data))
                fig.add_trace(
                    go.Scatter3d(
                        x=[initial_x], y=[initial_y], z=[1.65],
                        mode="markers", name="Delivery robot body",
                        marker={"size": 15, "color": "#168a85", "symbol": "square",
                                "line": {"color": "white", "width": 1.5}},
                        hovertemplate="Small delivery robot<extra></extra>",
                    )
                )
                robot_trace_indices.append(len(fig.data))
                fig.add_trace(
                    go.Scatter3d(
                        x=[initial_x], y=[initial_y], z=[2.65],
                        mode="markers", name="Robot head",
                        marker={"size": 11, "color": "#dce6f1", "symbol": "square",
                                "line": {"color": "#344054", "width": 1}},
                        showlegend=False, hoverinfo="skip",
                    )
                )
                robot_trace_indices.append(len(fig.data))
                fig.add_trace(
                    go.Scatter3d(
                        x=[initial_x - 0.7, initial_x + 0.7],
                        y=[initial_y - 0.15, initial_y - 0.15], z=[0.85, 0.85],
                        mode="markers", name="Robot wheels",
                        marker={"size": 5, "color": "#263238", "symbol": "circle"},
                        showlegend=False, hoverinfo="skip",
                    )
                )
                robot_trace_indices.append(len(fig.data))
                fig.add_trace(
                    go.Scatter3d(
                        x=[initial_x - 0.24, initial_x + 0.24],
                        y=[initial_y - 0.45, initial_y - 0.45], z=[2.68, 2.68],
                        mode="markers", name="Robot sensors",
                        marker={"size": 2.5, "color": "#263238"},
                        showlegend=False, hoverinfo="skip",
                    )
                )
                robot_trace_indices.append(len(fig.data))
                fig.add_trace(
                    go.Scatter3d(
                        x=[initial_x, initial_x, None],
                        y=[initial_y, initial_y, None], z=[3.0, 3.55, None],
                        mode="lines", name="Robot antenna",
                        line={"color": "#344054", "width": 3},
                        showlegend=False, hoverinfo="skip",
                    )
                )
                robot_trace_indices.append(len(fig.data))
                fig.add_trace(
                    go.Scatter3d(
                        x=[initial_x], y=[initial_y], z=[3.55],
                        mode="markers", name="Robot antenna light",
                        marker={"size": 5, "color": "#f2a900", "symbol": "circle"},
                        showlegend=False, hoverinfo="skip",
                    )
                )

                robot_frames = []
                for index, (robot_x, robot_y) in enumerate(positions):
                    robot_frames.append(
                        go.Frame(
                            name=str(index),
                            data=[
                                go.Scatter3d(x=[robot_x], y=[robot_y], z=[1.65]),
                                go.Scatter3d(x=[robot_x], y=[robot_y], z=[2.65]),
                                go.Scatter3d(
                                    x=[robot_x - 0.7, robot_x + 0.7],
                                    y=[robot_y - 0.15, robot_y - 0.15], z=[0.85, 0.85],
                                ),
                                go.Scatter3d(
                                    x=[robot_x - 0.24, robot_x + 0.24],
                                    y=[robot_y - 0.45, robot_y - 0.45], z=[2.68, 2.68],
                                ),
                                go.Scatter3d(
                                    x=[robot_x, robot_x, None],
                                    y=[robot_y, robot_y, None], z=[3.0, 3.55, None],
                                ),
                                go.Scatter3d(x=[robot_x], y=[robot_y], z=[3.55]),
                            ],
                            traces=robot_trace_indices,
                        )
                    )
                fig.frames = robot_frames
                fig.update_layout(
                    updatemenus=[
                        {
                            "type": "buttons",
                            "direction": "left",
                            "x": 0.02,
                            "y": 1.08,
                            "buttons": [
                                {
                                    "label": "▶ Play delivery",
                                    "method": "animate",
                                    "args": [None, {
                                        "frame": {"duration": 180, "redraw": True},
                                        "transition": {"duration": 0},
                                        "fromcurrent": True,
                                    }],
                                },
                                {
                                    "label": "⏸ Pause",
                                    "method": "animate",
                                    "args": [[None], {
                                        "mode": "immediate",
                                        "frame": {"duration": 0, "redraw": False},
                                        "transition": {"duration": 0},
                                    }],
                                },
                            ],
                        }
                    ]
                )

        fig.update_layout(
            title="DIT Campus — 3D Buildings, Priority Routes and Delivery Robot",
            width=1400,
            height=850,
            margin={"l": 0, "r": 0, "b": 0, "t": 65},
            legend={"orientation": "h", "y": -0.08},
            scene={
                "xaxis": {"title": "Campus X (m)", "range": [-22, 140], "backgroundcolor": "#f5f7fa"},
                "yaxis": {"title": "Campus Y (m)", "range": [-25, 105], "backgroundcolor": "#f5f7fa"},
                "zaxis": {"title": "Illustrative height (m)", "range": [0, 34], "backgroundcolor": "#f5f7fa"},
                "aspectmode": "manual",
                "aspectratio": {"x": 1, "y": 1, "z": 0.48},
                "camera": {"eye": {"x": 1.55, "y": 1.55, "z": 1.25}},
            },
        )
        fig.show(
            config={
                "responsive": True,
                "displaylogo": False,
                "toImageButtonOptions": {
                    "format": "png",
                    "filename": "dit_campus_delivery_model",
                    "width": 2000,
                    "height": 1400,
                    "scale": 2,
                },
            }
        )

    def add_to_cart(_):
        with message_output:
            clear_output(wait=True)
            if food.value is None:
                display(HTML("<span style='color:#b42318'>No menu item is available.</span>"))
                return
            food_id = int(food.value)
            cart[food_id] = cart.get(food_id, 0) + int(quantity.value)
            display(
                HTML(
                    f"<span style='color:#067647'>Added "
                    f"<b>{escape(menu_by_id[food_id]['name'])}</b> to the cart.</span>"
                )
            )
        refresh_cart()

    def remove_from_cart(_):
        with message_output:
            clear_output(wait=True)
            if food.value not in cart:
                display(HTML("<span style='color:#b54708'>That item is not in the cart.</span>"))
                return
            removed_name = menu_by_id[int(food.value)]["name"]
            del cart[int(food.value)]
            display(
                HTML(
                    f"<span style='color:#475467'>Removed "
                    f"<b>{escape(removed_name)}</b> from the cart.</span>"
                )
            )
        refresh_cart()

    def preview_route(_):
        with route_output:
            clear_output(wait=True)
            route, scenarios, alternatives = create_route_plan(
                cafeteria.value, destination.value, priority.value
            )
            if route is None:
                display(HTML("<b style='color:#b42318'>No route found.</b>"))
                return
            display(
                HTML(
                    route_report_html(
                        cafeteria.value, destination.value, priority.value,
                        scenarios, alternatives, "Priority route comparison",
                    )
                )
            )
            draw_campus_map(
                route,
                comparison_routes=scenarios,
                selected_priority=priority.value,
            )

    def show_history(_):
        with history_output:
            clear_output(wait=True)
            orders = database.list_orders()
            if not orders:
                display(HTML("<div style='padding:12px;color:#667085'>No saved orders yet.</div>"))
                return

            rows = []
            for order in orders:
                item_text = ", ".join(
                    f"{escape(item['name'])} × {item['quantity']}"
                    for item in order["items"]
                )
                rows.append(
                    "<tr>"
                    f"<td>#{order['order_id']}</td>"
                    f"<td>{escape(order['customer_name'])}</td>"
                    f"<td>{escape(order['cafeteria'])} → {escape(order['destination'])}</td>"
                    f"<td>{escape(item_text)}</td>"
                    f"<td>{escape(order['priority'])}</td>"
                    f"<td>{escape(order['status'])}</td>"
                    f"<td style='text-align:right'>₹{order['total']:.2f}</td>"
                    "</tr>"
                )
            table = (
                "<div style='overflow-x:auto'><table "
                "style='border-collapse:collapse;width:100%;font-size:13px'>"
                "<thead><tr style='background:#f2f6f8'>"
                "<th style='padding:8px'>Order</th><th>Customer</th>"
                "<th>Route</th><th>Items</th><th>Priority</th>"
                "<th>Status</th><th>Total</th></tr></thead><tbody>"
                + "".join(rows)
                + "</tbody></table></div>"
            )
            display(HTML(table))

    def place_order(_):
        place_button.disabled = True
        try:
            if active_order_id["value"] is not None:
                raise ValueError(
                    "Mark the current delivery complete before placing another order."
                )
            name = customer.value.strip()
            if not name:
                raise ValueError("Enter a customer or department name first.")
            if not cart:
                raise ValueError("Add at least one food item to the cart.")

            start = cafeteria.value
            goal = destination.value
            route, scenarios, alternatives = create_route_plan(
                start, goal, priority.value
            )
            if route is None:
                raise ValueError("No connected route exists for these locations.")

            order_id, total = database.create_order(
                customer_name=name,
                cafeteria=start,
                destination=goal,
                selections=list(cart.items()),
                priority=priority.value,
            )
            database.update_order_status(order_id, "PREPARING")

            with message_output:
                clear_output(wait=True)
                display(
                    HTML(
                        f"<div style='padding:12px;border-radius:10px;"
                        f"background:#eff8ff;color:#175cd3'>"
                        f"Order <b>#{order_id}</b> saved for "
                        f"<b>{escape(name)}</b>. Total: <b>₹{total:.2f}</b>. "
                        "Preparing the delivery simulation…</div>"
                    )
                )

            database.update_order_status(order_id, "OUT_FOR_DELIVERY")
            active_order_id["value"] = order_id
            mark_delivered_button.disabled = False
            with route_output:
                clear_output(wait=True)
                display(
                    HTML(
                        route_report_html(
                            start, goal, priority.value, scenarios, alternatives,
                            f"Order #{order_id} route comparison",
                        )
                    )
                )
                draw_campus_map(
                    route,
                    animate=True,
                    comparison_routes=scenarios,
                    selected_priority=priority.value,
                )

            with message_output:
                clear_output(wait=True)
                display(
                    HTML(
                        f"<div style='padding:12px;border-radius:10px;"
                        f"background:#eff8ff;color:#175cd3'>"
                        f"Order <b>#{order_id}</b> is saved and out for delivery. "
                        "Use the 3D viewer controls, then click Mark delivered.</div>"
                    )
                )
            cart.clear()
            refresh_cart()
            show_history(None)
        except Exception as error:
            with message_output:
                clear_output(wait=True)
                display(
                    HTML(
                        "<div style='padding:12px;border-radius:10px;"
                        "background:#fef3f2;color:#b42318'>"
                        f"{escape(str(error))}</div>"
                    )
                )
        finally:
            place_button.disabled = False

    def mark_delivered(_):
        order_id = active_order_id["value"]
        if order_id is None:
            return
        try:
            database.update_order_status(order_id, "DELIVERED")
            active_order_id["value"] = None
            mark_delivered_button.disabled = True
            with message_output:
                clear_output(wait=True)
                display(
                    HTML(
                        f"<div style='padding:12px;border-radius:10px;"
                        f"background:#ecfdf3;color:#067647'>"
                        f"Order <b>#{order_id}</b> marked as delivered.</div>"
                    )
                )
            show_history(None)
        except Exception as error:
            with message_output:
                clear_output(wait=True)
                display(
                    HTML(
                        "<div style='padding:12px;border-radius:10px;"
                        "background:#fef3f2;color:#b42318'>"
                        f"{escape(str(error))}</div>"
                    )
                )

    add_button.on_click(add_to_cart)
    remove_button.on_click(remove_from_cart)
    preview_button.on_click(preview_route)
    place_button.on_click(place_order)
    mark_delivered_button.on_click(mark_delivered)
    history_button.on_click(show_history)

    form_card = widgets.VBox(
        [
            widgets.HTML("<h3 style='margin:0 0 8px'>1. Delivery details</h3>"),
            customer,
            widgets.HBox([cafeteria, destination, priority]),
            widgets.HTML("<h3 style='margin:14px 0 8px'>2. Add food</h3>"),
            widgets.HBox([food, quantity]),
            widgets.HBox([add_button, remove_button]),
            widgets.HTML("<h3 style='margin:14px 0 8px'>Your cart</h3>"),
            cart_output,
            widgets.HBox([preview_button, place_button]),
            widgets.HBox([mark_delivered_button, history_button]),
            message_output,
        ],
        layout=widgets.Layout(
            width="100%",
            padding="18px",
            border="1px solid #d0d5dd",
        ),
    )

    route_card = widgets.VBox(
        [widgets.HTML("<h3 style='margin:0 0 8px'>3D buildings, priority routes, ETAs and robot</h3>"), route_output],
        layout=widgets.Layout(
            width="100%",
            padding="18px",
            border="1px solid #d0d5dd",
        ),
    )
    history_card = widgets.VBox(
        [
            widgets.HTML("<h3 style='margin:0 0 8px'>Saved orders</h3>"),
            history_output,
        ],
        layout=widgets.Layout(
            width="100%",
            padding="18px",
            border="1px solid #d0d5dd",
        ),
    )

    dashboard = widgets.VBox(
        [
            header,
            widgets.HTML(
                "<p style='color:#667085;margin:0'>Menu items: "
                f"<b>{len(menu_items)}</b> &nbsp; · &nbsp; "
                f"Campus locations: <b>{len(NODE_COORDS)}</b>"
                "</p>"
            ),
            form_card,
            route_card,
            history_card,
            widgets.HTML(
                "<small style='color:#667085'>This is a software "
                "simulation. Building footprints, heights, paths, and distances "
                "are illustrative sample data; replace them with measured campus "
                "data for an accurate model.</small>"
            ),
        ],
        layout=widgets.Layout(width="100%"),
    )

    display(dashboard)
    refresh_cart()
    show_history(None)

launch_gui()


In [ ]:
from google.colab import files
from pathlib import Path
db_path = Path.cwd() / "food_delivery.db"
print("Database exists:", db_path.exists())
if db_path.exists():
    files.download(str(db_path))
else:
    print("Run the order cell first to create the database.")


Database exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>